# 복지마을 방송국 (Welfare Village Broadcaster)
**LangGraph + Tavily 기반 멀티에이전트**

> "읽지 못해도, 스마트폰을 못 써도, 말 한마디면 복지 혜택을 받을 수 있게."

## 아키텍처
- **Supervisor 패턴**: LLM 라우터가 사용자 의도(`search` / `eligibility` / `broadcast` / `qna` / `done`)에 따라 서브 에이전트로 분기
- **서브 에이전트(노드)**
  1. `welfare_search` — Tavily로 복지로(bokjiro)·정부24 최신 복지 정보 검색
  2. `eligibility_check` — 사용자 프로필(나이/지역/소득/조건)로 자격 매칭 (규칙 기반 `@tool`)
  3. `easy_translate` — 행정 용어를 어르신 친화 말투로 변환 (사투리 옵션)
  4. `broadcast_script` — 마을 방송용 TTS 스크립트 생성
  5. `qna_agent` — 전화 상담 시뮬레이션 (음성 챗봇 페르소나)
- **State**: `messages`(add_messages), `user_profile`, `search_results`, `eligible_benefits`, `easy_text`, `broadcast_text`
- **메모리**: `InMemorySaver` (데모용, 운영 시 `SqliteSaver`로 교체)
- **관찰성**: LangFuse `CallbackHandler` (옵션)

## 시연 시나리오
1. 충청도 거주 78세 어르신 프로필 → 자격 가능 복지 검색
2. 행정 공지문 → 어르신 친화 사투리 방송 스크립트
3. 전화 상담: "나도 받을 수 있나?" 음성 챗봇 응답


## 1. 환경 설정 & API 키

필요한 API 키 (`.env` 파일에 작성):
- `OPENAI_API_KEY` — https://platform.openai.com/api-keys
- `TAVILY_API_KEY` — https://app.tavily.com (월 1,000회 무료)
- `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` (선택) — https://cloud.langfuse.com
- `LANGFUSE_HOST` (선택, 기본 `https://cloud.langfuse.com`)

> `.env.example`을 `.env`로 복사한 뒤 키를 채워넣으세요.


In [2]:
# 의존성 설치 (최초 1회)
!pip install -r requirements.txt


ERROR: Exception:
Traceback (most recent call last):
  File "C:\anaconda\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\anaconda\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "C:\anaconda\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "C:\anaconda\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ^^^^^^^^^^^^^^^^^^^
  File "C:\anaconda\Lib\http\client.py", line 479, in read
    s = self.fp.read(amt)
        ^^^^^^^^^^^^^^^^^
  File "C:\anaconda\Lib\socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\anaconda\Lib\ssl.py", line 1251, in recv_int

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(".env", override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 .env에 없습니다"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY가 .env에 없습니다"

# LangFuse는 선택 - 키 없으면 자동 비활성화
USE_LANGFUSE = bool(os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"))
print(f"LangFuse 사용: {USE_LANGFUSE}")


## 2. 모델 & 도구 정의

`init_chat_model`로 모델 초기화, `@tool`로 도구 정의. Tavily는 `TavilySearch` 도구를 사용합니다.


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field
from typing import Literal, Optional

# 메인 LLM (정확도 높음)
llm = init_chat_model("openai:gpt-4o-mini", temperature=0.3)
# 라우팅용 경량 LLM
router_llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

# Tavily 검색 도구 - 복지로/정부24 도메인 우선
tavily_tool = TavilySearch(
    max_results=5,
    topic="general",
    include_domains=["bokjiro.go.kr", "gov.kr", "mohw.go.kr", "korea.kr"],
    search_depth="advanced",
)
print("LLM, Tavily 초기화 완료")


In [ ]:
# ----- 자격 확인 도구 (규칙 기반 데모) -----
WELFARE_RULES = [
    {
        "name": "기초연금",
        "min_age": 65,
        "income_max": 2280000,  # 단독가구 선정기준액(2026 예시)
        "description": "만 65세 이상 소득하위 70% 어르신께 매월 최대 약 34만원 지급",
    },
    {
        "name": "노인일자리",
        "min_age": 60,
        "income_max": None,
        "description": "공공형/사회서비스형 노인일자리, 월 27~71만원 활동비",
    },
    {
        "name": "에너지바우처",
        "min_age": 65,
        "income_max": 1800000,
        "description": "겨울철 난방비, 여름철 냉방비 지원 (가구당 약 30만원)",
    },
    {
        "name": "장애인연금",
        "min_age": 18,
        "income_max": 1300000,
        "requires_disability": True,
        "description": "중증 장애인에게 매월 기초급여+부가급여 지급",
    },
    {
        "name": "농어촌 노인돌봄",
        "min_age": 65,
        "rural_only": True,
        "income_max": None,
        "description": "농어촌 거주 어르신 방문 돌봄·안부확인 서비스",
    },
]

class UserProfile(BaseModel):
    age: int = Field(description="만 나이")
    region: str = Field(description="거주 지역 (예: 충청남도 부여군)")
    monthly_income: Optional[int] = Field(default=None, description="월 소득(원)")
    has_disability: bool = Field(default=False, description="장애 등록 여부")
    is_rural: bool = Field(default=False, description="농어촌 거주 여부")

@tool
def check_eligibility(profile: UserProfile) -> list[dict]:
    """사용자 프로필을 받아 신청 가능한 복지 제도를 매칭한다.

    Args:
        profile: 사용자의 나이, 지역, 소득, 장애 여부, 농어촌 여부.
    Returns:
        자격이 맞는 복지 제도 리스트(이름·설명 포함).
    """
    matched = []
    for rule in WELFARE_RULES:
        if profile.age < rule["min_age"]:
            continue
        if rule.get("income_max") and profile.monthly_income and profile.monthly_income > rule["income_max"]:
            continue
        if rule.get("requires_disability") and not profile.has_disability:
            continue
        if rule.get("rural_only") and not profile.is_rural:
            continue
        matched.append({"name": rule["name"], "description": rule["description"]})
    return matched

print("도구 등록 완료:", [check_eligibility.name, tavily_tool.name])


## 3. State 정의

`TypedDict`로 그래프 전역 상태를 명시. `messages`만 `add_messages` 리듀서를 사용합니다 (덮어쓰기 방지).


In [ ]:
from typing import TypedDict, Annotated, List, Dict, Any
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class WelfareState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    user_profile: Dict[str, Any]          # {"age":..., "region":..., ...}
    search_results: List[Dict[str, Any]]  # Tavily 결과
    eligible_benefits: List[Dict[str, str]]  # 자격 매칭 결과
    easy_text: str                        # 쉬운 말 변환본
    broadcast_text: str                   # 방송 스크립트
    next_action: str                      # 라우터 결정값


## 4. 노드(에이전트) 함수 정의

각 노드는 `State -> dict` 형태의 부분 업데이트를 반환합니다.


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ---------- 4-1. Supervisor (라우터) ----------
class RouteDecision(BaseModel):
    next: Literal["search", "eligibility", "easy", "broadcast", "qna", "done"] = Field(
        description="다음에 실행할 서브에이전트"
    )
    reason: str = Field(description="결정 근거(한 줄)")

SUPERVISOR_PROMPT = """당신은 '복지마을 방송국' 시스템의 매니저입니다.
사용자 메시지와 현재 상태를 보고 다음 중 하나를 선택하세요:

- search: 최신 복지 제도/지원금 정보를 웹에서 찾아야 할 때 (Tavily)
- eligibility: 사용자가 본인의 자격(나이·소득·지역 등)으로 받을 수 있는 복지를 묻는 경우
- easy: 행정 공지문/딱딱한 텍스트를 어르신이 알아듣기 쉬운 말로 바꿔야 할 때
- broadcast: 마을 방송용 TTS 스크립트를 만들어야 할 때
- qna: 전화 상담처럼 자유 대화 응답이 필요할 때 (기본값에 가까움)
- done: 사용자의 요구가 충분히 해결되어 그래프를 종료해야 할 때

이미 자격 결과(eligible_benefits)나 쉬운 텍스트(easy_text)가 있고 사용자가 추가 작업을 요청하지 않았다면 done을 선택하세요."""

def supervisor_node(state: WelfareState) -> dict:
    structured = router_llm.with_structured_output(RouteDecision)
    last_msgs = state["messages"][-4:]  # 최근 4개만 컨텍스트
    summary = (
        f"이미 검색 완료: {len(state.get('search_results', []) or [])}건\n"
        f"자격 매칭: {len(state.get('eligible_benefits', []) or [])}건\n"
        f"쉬운 변환본: {'있음' if state.get('easy_text') else '없음'}\n"
        f"방송 스크립트: {'있음' if state.get('broadcast_text') else '없음'}"
    )
    msgs = [SystemMessage(content=SUPERVISOR_PROMPT),
            SystemMessage(content=f"현재 상태 요약:\n{summary}"),
            *last_msgs]
    decision = structured.invoke(msgs)
    return {"next_action": decision.next,
            "messages": [AIMessage(content=f"[Supervisor] 다음 단계: {decision.next} ({decision.reason})")]}


# ---------- 4-2. 복지 검색 에이전트 (Tavily) ----------
def search_node(state: WelfareState) -> dict:
    # 사용자 마지막 발화에서 검색 쿼리 추출
    last_user = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    query = last_user.content if last_user else "최신 복지 제도"
    # 프로필 키워드 추가
    profile = state.get("user_profile") or {}
    if profile.get("region"):
        query += f" {profile['region']}"
    if profile.get("age"):
        query += f" {profile['age']}세"

    raw = tavily_tool.invoke({"query": query})
    results = raw.get("results", []) if isinstance(raw, dict) else raw
    summary_lines = [f"- {r.get('title','')}: {r.get('url','')}" for r in results[:5]]
    return {
        "search_results": results,
        "messages": [AIMessage(content="🔎 복지 검색 결과:\n" + "\n".join(summary_lines))],
    }


# ---------- 4-3. 자격 확인 에이전트 ----------
def eligibility_node(state: WelfareState) -> dict:
    profile = state.get("user_profile") or {}
    if not profile.get("age"):
        return {"messages": [AIMessage(content="자격 확인을 위해 어르신의 나이/지역/소득 정보가 필요합니다.")]}
    up = UserProfile(**profile)
    matched = check_eligibility.invoke({"profile": up})
    if not matched:
        msg = "현재 입력하신 조건으로는 매칭되는 복지를 찾지 못했습니다. 추가 정보를 알려주시면 다시 확인해드리겠습니다."
    else:
        msg = "✅ 신청 가능 복지:\n" + "\n".join([f"- **{b['name']}**: {b['description']}" for b in matched])
    return {"eligible_benefits": matched, "messages": [AIMessage(content=msg)]}


# ---------- 4-4. 쉬운 말 변환 에이전트 ----------
EASY_PROMPT = """당신은 행정 공지를 시골 어르신도 알아듣게 풀어주는 친근한 동네 통장님입니다.
규칙:
1) 한자어/외래어 줄이기. (예: '도래하였으니' → '입니다')
2) 짧은 문장(15자 내외) 위주.
3) 따뜻하고 정중한 어투. 어르신 호칭 사용.
4) 신청 방법/장소/필요 서류를 또렷하게.
5) 사투리 옵션이 'chungcheong'이면 충청도 말투, 'jeolla'면 전라도, 그 외 표준어."""

def easy_translate_node(state: WelfareState) -> dict:
    # 변환할 원문 추출: 마지막 사용자 메시지
    last_user = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    if not last_user:
        return {"messages": [AIMessage(content="변환할 원문이 없습니다.")]}
    dialect = (state.get("user_profile") or {}).get("dialect", "standard")
    out = llm.invoke([SystemMessage(content=EASY_PROMPT + f"\n사투리 옵션: {dialect}"),
                      HumanMessage(content=f"다음 행정 공지를 변환:\n\n{last_user.content}")])
    return {"easy_text": out.content, "messages": [AIMessage(content=f"📣 쉬운 말 변환:\n{out.content}")]}


# ---------- 4-5. 방송 스크립트 생성 ----------
BROADCAST_PROMPT = """당신은 마을 스피커 방송용 멘트를 짜는 작가입니다.
구조 (반드시 이 순서):
1) 인삿말 ("어르신 안녕하세요, 마을 방송 알려드립니다.")
2) 핵심 안내 (한 문장)
3) 대상자 안내
4) 신청 방법 (장소·날짜·필요 서류)
5) 마무리 인사

분량: 30초 분량 (한국어 약 90-130자). 따뜻하고 천천히 읽을 수 있게."""

def broadcast_node(state: WelfareState) -> dict:
    base = state.get("easy_text") or ""
    benefits = state.get("eligible_benefits") or []
    last_user = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    src = base or (last_user.content if last_user else "")
    if benefits:
        src += "\n\n관련 복지: " + ", ".join([b["name"] for b in benefits])
    out = llm.invoke([SystemMessage(content=BROADCAST_PROMPT),
                      HumanMessage(content=f"원문/맥락:\n{src}")])
    return {"broadcast_text": out.content,
            "messages": [AIMessage(content=f"📻 마을 방송 스크립트:\n{out.content}")]}


# ---------- 4-6. Q&A 에이전트 (전화 상담 페르소나) ----------
QNA_PROMPT = """당신은 어르신께 전화로 복지를 안내하는 친절한 상담원입니다.
- 천천히, 짧은 문장으로.
- 어려운 용어는 그때그때 풀어 설명.
- 모르면 "면사무소에 전화 한 통 넣어드릴게요"라고 안내.
- 마지막에는 항상 "더 궁금하신 점 있으세요?"로 마무리.

자격 매칭 결과나 검색 결과가 state에 있으면 적극 활용하세요."""

def qna_node(state: WelfareState) -> dict:
    context = ""
    if state.get("eligible_benefits"):
        context += "\n자격 가능: " + ", ".join([b["name"] for b in state["eligible_benefits"]])
    if state.get("search_results"):
        titles = [r.get("title","") for r in state["search_results"][:3]]
        context += "\n검색결과 일부: " + " / ".join(titles)
    out = llm.invoke([SystemMessage(content=QNA_PROMPT + ("\n참고:" + context if context else "")),
                      *state["messages"][-6:]])
    return {"messages": [AIMessage(content=out.content)]}

print("6개 노드 정의 완료")


## 5. StateGraph 조립

Supervisor → conditional edge → 각 서브에이전트 → Supervisor 로 돌아오는 루프 구조.
`done`이 선택되면 `END`로 종료.


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

def route_from_supervisor(state: WelfareState) -> str:
    nxt = state.get("next_action", "qna")
    if nxt == "done":
        return END
    return {
        "search": "welfare_search",
        "eligibility": "eligibility_check",
        "easy": "easy_translate",
        "broadcast": "broadcast_script",
        "qna": "qna_agent",
    }.get(nxt, "qna_agent")

builder = StateGraph(WelfareState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("welfare_search", search_node)
builder.add_node("eligibility_check", eligibility_node)
builder.add_node("easy_translate", easy_translate_node)
builder.add_node("broadcast_script", broadcast_node)
builder.add_node("qna_agent", qna_node)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "welfare_search": "welfare_search",
        "eligibility_check": "eligibility_check",
        "easy_translate": "easy_translate",
        "broadcast_script": "broadcast_script",
        "qna_agent": "qna_agent",
        END: END,
    },
)
# 모든 서브에이전트 끝나면 supervisor로 복귀
for node in ["welfare_search", "eligibility_check", "easy_translate", "broadcast_script", "qna_agent"]:
    builder.add_edge(node, "supervisor")

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)
print("그래프 컴파일 완료")


## 6. 그래프 시각화

컴파일된 그래프를 그대로 평가하면 Jupyter에서 자동으로 토폴로지가 렌더링됩니다.


In [ ]:
graph


In [ ]:
# (옵션) PNG로 저장하고 싶을 때
try:
    png_bytes = graph.get_graph().draw_mermaid_png()
    with open("welfare_graph.png", "wb") as f:
        f.write(png_bytes)
    print("welfare_graph.png 저장 완료")
except Exception as e:
    print(f"PNG 렌더링 스킵: {e}")


## 7. LangFuse 콜백 (옵션)

LangFuse 키가 있으면 모든 노드 실행을 자동 기록합니다.


In [ ]:
langfuse_handler = None
if USE_LANGFUSE:
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print("LangFuse 핸들러 활성화")
else:
    print("LangFuse 키 없음 - 트레이싱 비활성화 (정상 동작에는 영향 없음)")

def cfg(thread_id: str):
    """invoke용 config 빌더"""
    c = {"configurable": {"thread_id": thread_id}}
    if langfuse_handler:
        c["callbacks"] = [langfuse_handler]
    return c


## 8. 시연 1 — "나도 받을 수 있나?" (자격 확인)

충청도 거주 78세 어르신 프로필로 자격 매칭.


In [ ]:
profile_grandma = {
    "age": 78,
    "region": "충청남도 부여군",
    "monthly_income": 800000,
    "has_disability": False,
    "is_rural": True,
    "dialect": "chungcheong",
}

initial = {
    "messages": [HumanMessage(content="이장님~ 나도 받을 수 있는 복지가 뭐가 있나 알아봐주세요.")],
    "user_profile": profile_grandma,
    "search_results": [],
    "eligible_benefits": [],
    "easy_text": "",
    "broadcast_text": "",
    "next_action": "",
}

result = graph.invoke(initial, config=cfg("demo-grandma-1"))

print("\n=== 최종 메시지 ===")
for m in result["messages"]:
    role = m.__class__.__name__.replace("Message","")
    print(f"[{role}] {m.content}\n")


## 9. 시연 2 — 행정 공지 → 어르신 친화 방송 스크립트

딱딱한 공지를 사투리 섞인 마을 방송으로 변환.


In [ ]:
official_notice = """기초연금 신청 기간이 도래하였으니, 만 65세 이상 거주민께서는 신분증 및 통장 사본을 지참하시어 6월 15일까지 행정복지센터를 방문 바랍니다. 미신청 시 해당 분기 지급이 누락될 수 있습니다."""

state2 = {
    "messages": [HumanMessage(content=official_notice)],
    "user_profile": {"dialect": "chungcheong"},
    "search_results": [],
    "eligible_benefits": [],
    "easy_text": "",
    "broadcast_text": "",
    "next_action": "",
}

# 사용자에게 두 단계가 필요함을 명시적으로 유도
state2["messages"].append(HumanMessage(content="이걸 어르신 알아듣게 풀어주시고, 마을 방송 멘트로도 만들어주세요."))

result2 = graph.invoke(state2, config=cfg("demo-broadcast-1"))

print("\n=== 쉬운 말 ===\n", result2.get("easy_text", "(없음)"))
print("\n=== 마을 방송 ===\n", result2.get("broadcast_text", "(없음)"))


## 10. 시연 3 — 전화 상담 멀티턴 (checkpointer 활용)

같은 `thread_id`로 두 번 호출 → 두 번째 호출에서 이전 컨텍스트가 자동 복원됨.


In [ ]:
thread = "demo-phone-1"

# 1번째 발화
r1 = graph.invoke(
    {"messages": [HumanMessage(content="여보세요? 나 올해 78인디, 에너지바우처라는거 받을 수 있나?")],
     "user_profile": {"age": 78, "monthly_income": 800000, "is_rural": True, "region": "충남 부여"},
     "search_results": [], "eligible_benefits": [], "easy_text": "", "broadcast_text": "", "next_action": ""},
    config=cfg(thread),
)
print("[1턴]", r1["messages"][-1].content, "\n")

# 2번째 발화 (이전 상태 자동 이어짐)
r2 = graph.invoke(
    {"messages": [HumanMessage(content="신청은 어디로 가야 하나요?")]},
    config=cfg(thread),
)
print("[2턴]", r2["messages"][-1].content)


## 11. 스트리밍 (옵션) — 노드 단위 진행 상황 출력


In [ ]:
for chunk in graph.stream(
    {"messages": [HumanMessage(content="우리 동네 어르신들한테 알려줄 최신 복지 뭐가 있나 검색해줘.")],
     "user_profile": {"region": "전라남도 진도군", "age": 72, "is_rural": True},
     "search_results": [], "eligible_benefits": [], "easy_text": "", "broadcast_text": "", "next_action": ""},
    config=cfg("demo-stream-1"),
    stream_mode="updates",
):
    for node, update in chunk.items():
        print(f"→ [{node}]")
        if "messages" in update and update["messages"]:
            print("  ", update["messages"][-1].content[:200], "...")


## 12. 다음 단계 (해커톤 확장 포인트)

1. **TTS 연동**: 노드 끝단에 `CLOVA Dubbing` / `ElevenLabs` 호출 추가 → mp3 다운로드
2. **STT 입력**: Whisper(`openai-whisper`)로 전화 음성 → 텍스트 → 그래프 invoke
3. **HITL 승인**: 실제 SMS/카톡 발송 노드 앞에 `HumanInTheLoopMiddleware` 추가
4. **장기 메모리**: `SqliteSaver` + `InMemoryStore`로 어르신별 프로필 영구 저장
5. **위험 감지**: 며칠간 응답 없는 어르신 자동 감지 → 복지사 알림 노드 추가
6. **MCP 서버화**: 복지 검색/자격 확인 도구를 FastMCP 서버로 분리 → 다른 에이전트에서도 재사용

### 발표용 한 줄
> **"복지는 신청하는 사람이 아니라, 필요한 사람에게 먼저 찾아가야 합니다."**
